# Customer segmentation: clustering
Rebuilds the RFM table from the raw workbook (the cleaning and RFM decisions are explained in
`eda.ipynb`), then scales, clusters, and profiles it.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = "../data/online_retail_II.xlsx"

sheets = pd.read_excel(DATA_PATH, sheet_name=None, engine="openpyxl")
list(sheets.keys())

['Year 2009-2010', 'Year 2010-2011']

In [2]:
# Last timestamp in 2009-2010. The 2010-2011 sheet repeats everything up to here,
# so only rows strictly after it are new.
OVERLAP_CUTOFF = sheets["Year 2009-2010"]["InvoiceDate"].max()

early = sheets["Year 2009-2010"]
late_new_rows = sheets["Year 2010-2011"][sheets["Year 2010-2011"]["InvoiceDate"] > OVERLAP_CUTOFF]

df = pd.concat([early, late_new_rows], ignore_index=True)

print(df.shape)
print(df.duplicated().sum())

(1044848, 8)
11812


In [3]:
print(df["Invoice"].map(type).value_counts())

df["Invoice"] = df["Invoice"].astype(str)

print(df["Invoice"].map(type).value_counts())

Invoice
<class 'int'>    1025677
<class 'str'>      19171
Name: count, dtype: int64
Invoice
<class 'str'>    1044848
Name: count, dtype: int64


In [4]:
# Cancellations and non-product codes are kept on purpose: the RFM build in Step 2 uses
# cancellations for Monetary only, and the non-product rows are -0.42% of value.
df_clean = df[df["Customer ID"].notna() & (df["Price"] > 0)].copy()

# Customer ID was a float only because of the nulls
df_clean["Customer ID"] = df_clean["Customer ID"].astype(int)

print(df_clean.shape)
print(df_clean["Customer ID"].nunique())

(809491, 8)
5939


In [5]:
SNAPSHOT_DATE = df_clean["InvoiceDate"].max() + pd.Timedelta(days=1)

purchases = df_clean[~df_clean["Invoice"].str.startswith("C")]

last_purchase = purchases.groupby("Customer ID")["InvoiceDate"].max()
recency = (SNAPSHOT_DATE - last_purchase).dt.days
recency.name = "Recency"

frequency = purchases.groupby("Customer ID")["Invoice"].nunique()
frequency.name = "Frequency"

line_total = df_clean["Quantity"] * df_clean["Price"]
monetary = line_total.groupby(df_clean["Customer ID"]).sum()
monetary.name = "Monetary"

rfm = pd.concat([recency, frequency, monetary], axis=1)

rfm = rfm.dropna()
rfm[["Recency", "Frequency"]] = rfm[["Recency", "Frequency"]].astype(int)

print(rfm.shape)

(5878, 3)
